# OpenGait Colab Notebook: Demo + CASIA-B Pretrained Validation

This notebook does the minimal end-to-end workflow:

1. Installs the required dependencies in Colab.
2. Clones the repository and applies the compatibility fixes needed for this setup.
3. Downloads the demo checkpoints and runs the demo on the bundled sample videos.
4. Downloads a CASIA-B Kaggle mirror into the correct place.
5. Preprocesses CASIA-B into OpenGait's pickle format.
6. Downloads the official CASIA-B pretrained checkpoints.
7. Runs pretrained-only validation for Baseline, GaitSet, and GaitGL.

Important notes:

- For the Kaggle download step, you must upload your own `kaggle.json` API key.
- You must set `KAGGLE_DATASET` to the CASIA-B Kaggle dataset slug you want to use.
- This notebook is written for a single Colab GPU, so the validation runs use `--nproc_per_node=1`.


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/jdyjjj/All-in-One-Gait.git"
WORKDIR = Path("/content")
REPO_ROOT = WORKDIR / "All-in-One-Gait" / "OpenGait"

!nvidia-smi || true
!python --version
!pip -q install --upgrade pip setuptools wheel
!pip -q install cython cython-bbox==0.1.3 filelock filterpy h5py kornia lap loguru motmetrics ninja onnx onnx-simplifier onnxoptimizer onnxruntime opencv-python==4.6.0.66 prettytable pyyaml scikit-image scikit-learn scipy tabulate tensorboard thop tqdm "visualdl>=2.2.0" paddlepaddle==2.6.2 gdown kaggle

if not (WORKDIR / "All-in-One-Gait").exists():
    !git clone {REPO_URL} {WORKDIR / 'All-in-One-Gait'}

%cd /content/All-in-One-Gait/OpenGait
!pip -q install -e demo/libs
!python - <<'PY'
import torch, cv2, paddle, yaml, numpy
print('torch:', torch.__version__)
print('torch cuda available:', torch.cuda.is_available())
print('opencv:', cv2.__version__)
print('paddle:', paddle.__version__)
print('numpy:', numpy.__version__)
PY


In [ ]:
from pathlib import Path

def replace_once(path, old, new):
    path = Path(path)
    text = path.read_text()
    if old in text:
        text = text.replace(old, new)
        path.write_text(text)
        print(f'patched: {path}')
    else:
        print(f'skipped (pattern not found): {path}')

# base_model.py fixes: local save_path + correct checkpoint restore path
replace_once(
    REPO_ROOT / 'opengait/modeling/base_model.py',
    """        # self.save_path = osp.join('output/', cfgs['data_cfg']['dataset_name'],\n        #                           cfgs['model_cfg']['model'], self.engine_cfg['save_name'])\n        # self.save_path = \"/home/jdy/Gaitdateset/gait_model/Baseline-150000.pt\"\n        self.save_path = \"/home/jdy/Gaitdateset/gait_model/Baseline-60000.pt\"\n""",
    """        self.save_path = osp.join(\n            'output/',\n            cfgs['data_cfg']['dataset_name'],\n            cfgs['model_cfg']['model'],\n            self.engine_cfg['save_name'],\n        )\n"""
)
replace_once(
    REPO_ROOT / 'opengait/modeling/base_model.py',
    """        # self._load_ckpt(save_name)\n        self._load_ckpt(self.save_path)\n""",
    """        self._load_ckpt(save_name)\n"""
)

# msg_manager.py fix: Pillow >= 10 ANTIALIAS compatibility
replace_once(
    REPO_ROOT / 'opengait/utils/msg_manager.py',
    "from time import strftime, localtime\n\nfrom torch.utils.tensorboard import SummaryWriter\n",
    "from time import strftime, localtime\nfrom PIL import Image\n\nfrom torch.utils.tensorboard import SummaryWriter\n"
)
replace_once(
    REPO_ROOT / 'opengait/utils/msg_manager.py',
    "import logging\n\n\nclass MessageManager:\n",
    "import logging\n\n\nif not hasattr(Image, 'ANTIALIAS') and hasattr(Image, 'Resampling'):\n    Image.ANTIALIAS = Image.Resampling.LANCZOS\n\n\nclass MessageManager:\n"
)

# common.py fixes: do not force CUDA for tensor conversion helpers
replace_once(
    REPO_ROOT / 'opengait/utils/common.py',
    """def ts2var(x, **kwargs):\n    return autograd.Variable(x, **kwargs).cuda()\n""",
    """def ts2var(x, **kwargs):\n    var = autograd.Variable(x, **kwargs)\n    if torch.cuda.is_available():\n        return var.cuda()\n    return var\n"""
)
replace_once(
    REPO_ROOT / 'opengait/utils/common.py',
    """def MeanIOU(msk1, msk2, eps=1.0e-9):\n    if not is_tensor(msk1):\n        msk1 = torch.from_numpy(msk1).cuda()\n    if not is_tensor(msk2):\n        msk2 = torch.from_numpy(msk2).cuda()\n""",
    """def MeanIOU(msk1, msk2, eps=1.0e-9):\n    if not is_tensor(msk1):\n        msk1 = torch.from_numpy(msk1)\n    if not is_tensor(msk2):\n        msk2 = torch.from_numpy(msk2)\n    if torch.cuda.is_available():\n        msk1 = msk1.cuda()\n        msk2 = msk2.cuda()\n"""
)

# baselineDemo.py fixes: allow CPU fallback for map_location/device logic
replace_once(
    REPO_ROOT / 'demo/libs/model/baselineDemo.py',
    """        self.device = torch.cuda.current_device()\n        torch.cuda.set_device(self.device)\n        self.to(device=torch.device(\n            \"cuda\", self.device))\n""",
    """        if torch.cuda.is_available():\n            self.device = torch.cuda.current_device()\n            torch.cuda.set_device(self.device)\n            target_device = torch.device(\"cuda\", self.device)\n        else:\n            self.device = torch.device(\"cpu\")\n            target_device = self.device\n        self.to(device=target_device)\n"""
)
replace_once(
    REPO_ROOT / 'demo/libs/model/baselineDemo.py',
    """        checkpoint = torch.load(save_name, map_location=torch.device(\n            \"cuda\"))\n""",
    """        map_location = torch.device(\"cuda\") if torch.cuda.is_available() else torch.device(\"cpu\")\n        checkpoint = torch.load(save_name, map_location=map_location)\n"""
)

# paddle infer fix: remove deprecated paddle.fluid.core usage
replace_once(
    REPO_ROOT / 'demo/libs/paddle/infer.py',
    """import paddle.fluid.core as core\ntrt_precision_map = {\n    \"int8\": core.AnalysisConfig.Precision.Int8,\n    \"fp32\": core.AnalysisConfig.Precision.Float32,\n    \"fp16\": core.AnalysisConfig.Precision.Half\n}\n\n""",
    """"""
)

# track.py fixes: automatic device choice, safe fp16, predictor cuda flag
replace_once(
    REPO_ROOT / 'demo/libs/track.py',
    '    "device": "gpu",\n',
    '    "device": "gpu" if torch.cuda.is_available() else "cpu",\n'
)
replace_once(
    REPO_ROOT / 'demo/libs/track.py',
    """    logger.info(\"\\tFusing model...\")\n    model = fuse_model(model)\n    model = model.half()\n    return model\n""",
    """    logger.info(\"\\tFusing model...\")\n    model = fuse_model(model)\n    if device.type == \"cuda\":\n        model = model.half()\n    return model\n"""
)
replace_once(
    REPO_ROOT / 'demo/libs/track.py',
    '    predictor = Predictor(model, exp, trt_file, decoder, device, True)\n',
    '    predictor = Predictor(model, exp, trt_file, decoder, device, device.type == "cuda")\n'
)

print('Patching complete.')


In [ ]:
%cd /content/All-in-One-Gait/OpenGait
!mkdir -p demo/checkpoints/gait_model demo/checkpoints/seg_model
!curl -L https://github.com/ShiqiYu/OpenGait/releases/download/v2.0/pretrained_grew_gaitbase.zip -o demo/checkpoints/gait_model/pretrained_grew_gaitbase.zip
!unzip -o -j demo/checkpoints/gait_model/pretrained_grew_gaitbase.zip -d demo/checkpoints/gait_model
!curl -L https://paddleseg.bj.bcebos.com/dygraph/pp_humanseg_v2/human_pp_humansegv2_mobile_192x192_inference_model_with_softmax.zip -o demo/checkpoints/seg_model/human_pp_humansegv2_mobile_192x192_inference_model_with_softmax.zip
!unzip -o demo/checkpoints/seg_model/human_pp_humansegv2_mobile_192x192_inference_model_with_softmax.zip -d demo/checkpoints/seg_model
!ls -lah demo/checkpoints/gait_model
!ls -lah demo/checkpoints/seg_model/human_pp_humansegv2_mobile_192x192_inference_model_with_softmax


In [ ]:
%cd /content/All-in-One-Gait/OpenGait
!python demo/libs/main.py


## CASIA-B Kaggle download

Set `KAGGLE_DATASET` below to the Kaggle dataset slug you want to use.

Expected raw format for OpenGait pretreatment:

```text
subject / sequence-type / view / frame.png
```

If your Kaggle mirror extracts one extra top-level folder, the helper cell below will try to locate the actual CASIA-B root automatically.


In [ ]:
from google.colab import files
import os
from pathlib import Path

KAGGLE_DATASET = 'REPLACE_WITH_YOUR_KAGGLE_DATASET_SLUG'
KAGGLE_DIR = Path('/root/.config/kaggle')
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)

print('Upload your kaggle.json file now...')
uploaded = files.upload()
assert 'kaggle.json' in uploaded, 'Please upload kaggle.json'
(KAGGLE_DIR / 'kaggle.json').write_bytes(uploaded['kaggle.json'])
os.chmod(KAGGLE_DIR / 'kaggle.json', 0o600)

RAW_DOWNLOAD_DIR = Path('/content/casiab_kaggle_download')
RAW_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

if KAGGLE_DATASET == 'REPLACE_WITH_YOUR_KAGGLE_DATASET_SLUG':
    raise ValueError('Edit KAGGLE_DATASET first.')

!kaggle datasets download -d {KAGGLE_DATASET} -p {RAW_DOWNLOAD_DIR} --unzip
!find {RAW_DOWNLOAD_DIR} -maxdepth 3 | head -200


In [ ]:
from pathlib import Path

def looks_like_casiab_raw(root: Path) -> bool:
    pngs = list(root.rglob('*.png'))
    if not pngs:
        return False
    for p in pngs[:50]:
        parts = p.parts
        if len(parts) < 4:
            continue
        parent = p.parent
        if len(parent.parts) >= 3:
            sid = parent.parent.parent.name
            seq = parent.parent.name
            view = parent.name
            if sid.isdigit() and '-' in seq and view.isdigit():
                return True
    return False

candidates = [RAW_DOWNLOAD_DIR] + [p for p in RAW_DOWNLOAD_DIR.rglob('*') if p.is_dir()]
matches = [p for p in candidates if looks_like_casiab_raw(p)]
assert matches, 'Could not locate a CASIA-B raw root automatically. Inspect RAW_DOWNLOAD_DIR and set CASIA_RAW_ROOT manually.'
CASIA_RAW_ROOT = matches[0]
CASIA_PKL_ROOT = Path('/content/casiab-pkl')
print('Detected CASIA_RAW_ROOT =', CASIA_RAW_ROOT)


In [ ]:
%cd /content/All-in-One-Gait/OpenGait
!python datasets/pretreatment.py --input_path {CASIA_RAW_ROOT} --output_path {CASIA_PKL_ROOT} --dataset CASIAB --img_size 64 --n_workers 4
!find {CASIA_PKL_ROOT} -maxdepth 4 | head -50


In [ ]:
%cd /content/All-in-One-Gait/OpenGait
!mkdir -p pretrained_assets output
!curl -L https://github.com/ShiqiYu/OpenGait/releases/download/v1.0/pretrained_casiab_model.zip -o pretrained_assets/pretrained_casiab_model.zip
!unzip -o pretrained_assets/pretrained_casiab_model.zip -d output
!find output/CASIA-B -maxdepth 5 -type f | sort


In [ ]:
from pathlib import Path

cfg_dir = REPO_ROOT / 'configs' / 'colab'
cfg_dir.mkdir(parents=True, exist_ok=True)

(cfg_dir / 'baseline_casiab_pretrained_colab.yaml').write_text(f'''data_cfg:\n  dataset_name: CASIA-B\n  dataset_root: {CASIA_PKL_ROOT}\n  dataset_partition: /content/All-in-One-Gait/OpenGait/datasets/CASIA-B/CASIA-B_include_005.json\n  num_workers: 2\n  cache: false\n  remove_no_gallery: false\n  test_dataset_name: CASIA-B\n\nevaluator_cfg:\n  enable_float16: true\n  restore_ckpt_strict: true\n  restore_hint: 60000\n  save_name: Baseline\n  eval_func: evaluate_indoor_dataset\n  sampler:\n    batch_shuffle: false\n    batch_size: 16\n    sample_type: all_ordered\n    type: InferenceSampler\n    frames_all_limit: 720\n  metric: euc\n  transform:\n    - type: BaseSilCuttingTransform\n      img_w: 64\n  cross_view_gallery: false\n\nloss_cfg:\n  - loss_term_weight: 1.0\n    margin: 0.2\n    type: TripletLoss\n    log_prefix: triplet\n  - loss_term_weight: 0.1\n    scale: 16\n    type: CrossEntropyLoss\n    log_prefix: softmax\n    log_accuracy: true\n\nmodel_cfg:\n  model: Baseline\n  backbone_cfg:\n    in_channels: 1\n    layers_cfg:\n      - BC-64\n      - BC-64\n      - M\n      - BC-128\n      - BC-128\n      - M\n      - BC-256\n      - BC-256\n    type: Plain\n  SeparateFCs:\n    in_channels: 256\n    out_channels: 256\n    parts_num: 31\n  SeparateBNNecks:\n    class_num: 74\n    in_channels: 256\n    parts_num: 31\n  bin_num:\n    - 16\n    - 8\n    - 4\n    - 2\n    - 1\n\noptimizer_cfg:\n  lr: 0.1\n  momentum: 0.9\n  solver: SGD\n  weight_decay: 0.0005\n\nscheduler_cfg:\n  gamma: 0.1\n  milestones:\n    - 20000\n    - 40000\n  scheduler: MultiStepLR\n\ntrainer_cfg:\n  enable_float16: true\n  fix_BN: false\n  log_iter: 100\n  with_test: true\n  restore_ckpt_strict: true\n  restore_hint: 0\n  save_iter: 10000\n  save_name: Baseline\n  sync_BN: false\n  total_iter: 60000\n  sampler:\n    batch_shuffle: true\n    batch_size:\n      - 8\n      - 16\n    frames_num_fixed: 30\n    frames_num_max: 50\n    frames_num_min: 25\n    sample_type: fixed_unordered\n    type: TripletSampler\n  transform:\n    - type: BaseSilCuttingTransform\n      img_w: 64\n''')

(cfg_dir / 'gaitset_casiab_pretrained_colab.yaml').write_text(f'''data_cfg:\n  dataset_name: CASIA-B\n  dataset_root: {CASIA_PKL_ROOT}\n  dataset_partition: /content/All-in-One-Gait/OpenGait/datasets/CASIA-B/CASIA-B_include_005.json\n  num_workers: 2\n  cache: false\n  remove_no_gallery: false\n  test_dataset_name: CASIA-B\n\nevaluator_cfg:\n  enable_float16: true\n  restore_ckpt_strict: true\n  restore_hint: 40000\n  save_name: GaitSet\n  eval_func: evaluate_indoor_dataset\n  sampler:\n    batch_size: 16\n    sample_type: all_ordered\n    type: InferenceSampler\n  metric: euc\n  cross_view_gallery: false\n\nloss_cfg:\n  loss_term_weight: 1.0\n  margin: 0.2\n  type: TripletLoss\n  log_prefix: triplet\n\nmodel_cfg:\n  model: GaitSet\n  in_channels:\n    - 1\n    - 32\n    - 64\n    - 128\n  SeparateFCs:\n    in_channels: 128\n    out_channels: 256\n    parts_num: 62\n  bin_num:\n    - 16\n    - 8\n    - 4\n    - 2\n    - 1\n\noptimizer_cfg:\n  lr: 0.1\n  momentum: 0.9\n  solver: SGD\n  weight_decay: 0.0005\n\nscheduler_cfg:\n  gamma: 0.1\n  milestones:\n    - 10000\n    - 20000\n    - 30000\n  scheduler: MultiStepLR\n\ntrainer_cfg:\n  enable_float16: true\n  log_iter: 100\n  with_test: true\n  restore_ckpt_strict: true\n  restore_hint: 0\n  save_iter: 10000\n  save_name: GaitSet\n  sync_BN: false\n  total_iter: 40000\n  sampler:\n    batch_shuffle: false\n    batch_size:\n      - 8\n      - 16\n    frames_num_fixed: 30\n    frames_num_max: 50\n    frames_num_min: 25\n    sample_type: fixed_unordered\n    type: TripletSampler\n''')

(cfg_dir / 'gaitgl_casiab_pretrained_colab.yaml').write_text(f'''data_cfg:\n  dataset_name: CASIA-B\n  dataset_root: {CASIA_PKL_ROOT}\n  dataset_partition: /content/All-in-One-Gait/OpenGait/datasets/CASIA-B/CASIA-B_include_005.json\n  num_workers: 2\n  cache: false\n  remove_no_gallery: false\n  test_dataset_name: CASIA-B\n\nevaluator_cfg:\n  enable_float16: true\n  restore_ckpt_strict: true\n  restore_hint: 80000\n  save_name: GaitGL\n  eval_func: evaluate_indoor_dataset\n  sampler:\n    batch_size: 1\n    sample_type: all_ordered\n    type: InferenceSampler\n  metric: euc\n  cross_view_gallery: false\n\nloss_cfg:\n  - loss_term_weight: 1.0\n    margin: 0.2\n    type: TripletLoss\n    log_prefix: triplet\n  - loss_term_weight: 1.0\n    scale: 1\n    type: CrossEntropyLoss\n    log_accuracy: true\n    label_smooth: false\n    log_prefix: softmax\n\nmodel_cfg:\n  model: GaitGL\n  channels:\n    - 32\n    - 64\n    - 128\n  class_num: 74\n\noptimizer_cfg:\n  lr: 1.0e-4\n  solver: Adam\n  weight_decay: 5.0e-4\n\nscheduler_cfg:\n  gamma: 0.1\n  milestones:\n    - 70000\n  scheduler: MultiStepLR\n\ntrainer_cfg:\n  enable_float16: true\n  with_test: true\n  log_iter: 100\n  restore_ckpt_strict: true\n  restore_hint: 0\n  save_iter: 10000\n  save_name: GaitGL\n  sync_BN: true\n  total_iter: 80000\n  sampler:\n    batch_shuffle: true\n    batch_size:\n      - 8\n      - 8\n    frames_num_fixed: 30\n    frames_skip_num: 0\n    sample_type: fixed_ordered\n    type: TripletSampler\n''')

print('Wrote configs to', cfg_dir)
!find /content/All-in-One-Gait/OpenGait/configs/colab -maxdepth 1 -type f -print


In [ ]:
%cd /content/All-in-One-Gait/OpenGait
!CUDA_VISIBLE_DEVICES=0 python -m torch.distributed.launch --master_port 29511 --nproc_per_node=1 opengait/main.py --cfgs ./configs/colab/baseline_casiab_pretrained_colab.yaml --phase test --iter 60000


In [ ]:
%cd /content/All-in-One-Gait/OpenGait
!CUDA_VISIBLE_DEVICES=0 python -m torch.distributed.launch --master_port 29512 --nproc_per_node=1 opengait/main.py --cfgs ./configs/colab/gaitset_casiab_pretrained_colab.yaml --phase test --iter 40000


In [ ]:
%cd /content/All-in-One-Gait/OpenGait
!CUDA_VISIBLE_DEVICES=0 python -m torch.distributed.launch --master_port 29513 --nproc_per_node=1 opengait/main.py --cfgs ./configs/colab/gaitgl_casiab_pretrained_colab.yaml --phase test --iter 80000


At this point you should have:

- the demo run completed on the sample videos,
- CASIA-B preprocessed into OpenGait pickle format,
- official CASIA-B pretrained checkpoints downloaded,
- pretrained validation metrics printed for Baseline, GaitSet, and GaitGL.
